In [ ]:
%pip install -qU langchain langchain-google-genai langchain_community tavily-python aiosqlite langchain-tavily
%pip install -qU langgraph langgraph-checkpoint-sqlite

In [ ]:
from typing import TypedDict, List

In [ ]:
class AgentState(TypedDict):
    task: str
    plan: str
    draft: str
    critique: str
    content: List[str]
    revision_number: int
    max_revisions: int

Creando los prompts para los agentes

In [ ]:
PLAN_PROMPT = """Eres un escritor especialista con la tarea de crear un esquema de alto nivel para una redacción. \
Escribe este esquema para el tema proporcionado por el usuario. Presenta un plan de la redacción junto con cualesquiera notas \
o instrucciones relevantes para las secciones."""

In [ ]:
WRITER_PROMPT = """Eres un asistente de redacción con la tarea de escribir excelentes redacciones de 5 párrafos. \
Genera la mejor redacción posible para la solicitud del usuario y el esquema inicial. \
Si el usuario proporciona críticas, responde con una versión revisada de tus intentos anteriores. \
Utiliza toda la información a continuación según sea necesario:

------------

{content}"""

Definiendo los prompts de reflexión e investigación


In [ ]:
REFLECTION_PROMPT = """Eres un profesor corrigiendo una redacción presentada. \
Genera una crítica y recomendaciones para la entrega del usuario. \
Proporciona recomendaciones detalladas, incluyendo solicitudes sobre extensión, profundidad, estilo, etc."""

In [ ]:
RESEARCH_PLAN_PROMPT = """Eres un investigador encargado de proporcionar información que puede \
ser utilizada al escribir la siguiente redacción. Genera una lista de consultas de investigación que \
recopilen cualquier información relevante. Genera como máximo 3 consultas."""

In [ ]:
RESEARCH_CRITIQUE_PROMPT = """Eres un investigador encargado de proporcionar información que puede \
ser utilizada al realizar cualquier revisión solicitada (según se describe a continuación). \
Genera una lista de consultas de investigación que recopilen cualquier información relevante. Genera \
como máximo 3 consultas."""

Definiendo la clase Queries


In [ ]:
from pydantic import BaseModel

In [ ]:
class Queries(BaseModel):
    queries: List[str]

In [ ]:
from tavily import TavilyClient
import os
tavily = TavilyClient(api_key=TAVILY_API_KEY)

In [ ]:
def plan_node(state: AgentState):
    messages = [
        SystemMessage(content=PLAN_PROMPT),
        HumanMessage(content=state['task'])
    ]
    response = model.invoke(messages)
    return {"plan": response.content}

Desarrollando el nodo de investigación


In [ ]:
def research_plan_node(state: AgentState):
    queries = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_PLAN_PROMPT),
        HumanMessage(content=state['task'])
    ])
    content = state['content'] or []
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])
    return {"content": content}

Creando el nodo de generación


In [ ]:
def generation_node(state: AgentState):
    content = "\n\n".join(state['content'] or [])
    user_message = HumanMessage(
        content=f"{state['task']}\n\nHere is my plan:\n\n{state['plan']}"
    )
    messages = [
        SystemMessage(
            content=WRITER_PROMPT.format(content=content)
        ),
        user_message
    ]
    response = model.invoke(messages)
    return {
        "draft": response.content,
        "revision_number": state.get("revision_number", 1) + 1
    }

Implementando el nodo de reflexión


In [ ]:
def reflection_node(state: AgentState):
    messages = [
        SystemMessage(content=REFLECTION_PROMPT),
        HumanMessage(content=state['draft'])
    ]
    response = model.invoke(messages)
    return {"critique": response.content}

Determinando la continuación del proceso


In [ ]:
def should_continue(state):
    if state["revision_number"] > state["max_revisions"]:
        return END
    return "reflect"

## 03 Orquestando MultiAgentes - Grafo

In [ ]:
builder = StateGraph(AgentState)

Añadiendo nodos y configurando el punto de entrada


In [ ]:
builder.add_node("planner", plan_node)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("research_critique", research_critique_node)

In [ ]:
builder.set_entry_point("planner")

Estableciendo aristas condicionales y secuenciales


In [ ]:
builder.add_conditional_edges(
    "generate",
    should_continue,
    {END: END, "reflect": "reflect"}
)

In [ ]:
builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generate")
builder.add_edge("reflect", "research_critique")
builder.add_edge("research_critique", "generate")

Compilando y visualizando el grafo


In [ ]:
graph = builder.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image, display, Markdown
import os

print("\n--- Tratando de Generar el PNG del Grafo vía Mermaid (Requiere Playwright!) ---")
try:
    image_data = graph.get_graph().draw_mermaid_png()
    display(Image(data=image_data))
    print("¡Grafo PNG generado y exhibido con éxito!")

except Exception as e:
    print(f"\nError al tratar de generar el PNG del grafo: {e}")
    print("Esto posiblemente se debe a que:")
    print("1. El método `.draw_mermaid_png()` no existe en su versión de LangGraph.")
    print("2. Faltan dependencias como 'playwright' o sus drivers no fueron instalados.")
    print("   Trata con: pip install playwright && playwright install")
    print("3. Otro error inesperado al acceder al grafo o renderizar.")

    print("\n--- Tratando generar únicamente el código Mermaid (Fallback) ---")
    try:
        mermaid_code = graph.get_graph().draw_mermaid()
        print("\n--- Código Mermaid Generado (Pégalo en https://mermaid.live/ o utiliza Markdown en un entorno compatible) ---")
        print(mermaid_code)
    
    except Exception as e_mermaid:
        print(f"Error al generar el código Mermaid: {e_mermaid}")
        print("Verifica si `graph.get_graph()` está correcto y es accesible.")

Ejecutando el flujo de trabajo


In [ ]:
thread = {"configurable": {"thread_id": "5"}}
for s in graph.stream({
    "task": "Cuál es la diferencia entre LangChain y LangSmith",
    "max_revisions": 2,
    "revision_number": 1,
    "content": [],
}, thread):
    print(s)

## 04 Orquestando MultiAgentes - Gradio

Configurando el entorno y definiendo clases


In [ ]:
# new_backend.py
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List
import operator
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from tavily import TavilyClient
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
import warnings
warnings.filterwarnings("ignore", message=".*TqdmWarning.*")

# Carga las variables de entorno desde el archivo .env
load_dotenv()

# Define las variables de entorno
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [ ]:
# Define el estado del agente (AgentState)
class AgentState(TypedDict):
    task: str
    plan: str
    draft: str
    critique: str
    content: List[str]
    revision_number: int
    max_revisions: int

# Define el modelo Pydantic para la salida estructurada
class Queries(BaseModel):
    queries: List[str]

# Inicializa la base de datos para los checkpoints
conn = sqlite3.connect("checkpoints.db", check_same_thread=False)
memory = SqliteSaver(conn)

Inicializando el modelo de lenguaje y definiendo prompts


In [ ]:
# Inicializa el modelo de lenguaje
model = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0, GOOGLE_API_KEY=GEMINI_API_KEY)

# Crea un Runnable para la salida estructurada (forma correcta para Gemini)
structured_model = model.with_structured_output(Queries)

# Prompts
PLAN_PROMPT = """Eres un escritor especialista con la tarea de crear un esquema de alto nivel para una redacción. \
Escribe este esquema para el tema proporcionado por el usuario. Presenta un plan de la redacción junto con una lista de consultas de búsqueda \
o instrucción relevante para las secciones."""

WRITER_PROMPT = """Eres un asistente de redacción con la tarea de escribir excelentes redacciones de 5 párrafos. \
Genera la mejor redacción posible para la solicitud del usuario y el esquema inicial. \
Si el usuario proporciona críticas, responde con una versión revisada de tus intentos anteriores. \
Utiliza toda la información a continuación según sea necesario: 

{content}"""

REFLECTION_PROMPT = """Eres un profesor encargado de evaluar un ensayo presentado. \
Genera una crítica detallada y recomendaciones para la entrega del usuario. \
Proporciona observaciones específicas, incluyendo sugerencias sobre extensión, profundidad, estilo, claridad y estructura."""

RESEARCH_PLAN_PROMPT = """Eres un investigador encargado de proporcionar información que pueda ser utilizada \
para redactar el siguiente ensayo. Genera una lista de consultas de búsqueda que permitan recopilar \
toda la información relevante. Genera como máximo 3 consultas."""

RESEARCH_CRITIQUE_PROMPT = """Eres un investigador encargado de proporcionar información que pueda ser utilizada \
para realizar las revisiones solicitadas (según se describe a continuación). \
Genera una lista de consultas de búsqueda que permitan recopilar \
toda la información relevante. Genera como máximo 3 consultas."""

# Inicializa el cliente Tavily
tavily = TavilyClient(api_key=TAVILY_API_KEY)

Construyendo el grafo y definiendo nodos


In [ ]:
# Definición de los nodos del LangGraph
def plan_node(state: AgentState):
    messages = [
        SystemMessage(content=PLAN_PROMPT), 
        HumanMessage(content=state['task'])
    ]
    response = model.invoke(messages)
    return {"plan": response.content}

def research_plan_node(state: AgentState):
    queries = structured_model.invoke([
        SystemMessage(content=RESEARCH_PLAN_PROMPT),
        HumanMessage(content=state['task'])
    ])
    content = state['content'] or []
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])
    return {"content": content}

def generation_node(state: AgentState):
    content = "\n\n".join(state['content'] or [])
    user_message = HumanMessage(
        content=f"{state['task']}\n\nHere is my plan:\n\n{state['plan']}")
    messages = [
        SystemMessage(
            content=WRITER_PROMPT.format(content=content)
        ),
        user_message
        ]
    response = model.invoke(messages)
    return {
        "draft": response.content, 
        "revision_number": state.get("revision_number", 0) + 1
    }

def reflection_node(state: AgentState):
    messages = [
        SystemMessage(content=REFLECTION_PROMPT), 
        HumanMessage(content=state['draft'])
    ]
    response = model.invoke(messages)
    return {"critique": response.content}

def research_critique_node(state: AgentState):
    queries = structured_model.invoke([
        SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
        HumanMessage(content=state['critique'])
    ])
    content = state['content'] or []
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            content.append(r['content'])
    return {"content": content}

def should_continue(state):
    if state["revision_number"] > state["max_revisions"]:
        return END
    return "reflect"

# Construcción del Grafo
builder = StateGraph(AgentState)

builder.add_node("planner", plan_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)
builder.add_node("research_critique", research_critique_node)

builder.set_entry_point("planner")

builder.add_conditional_edges(
    "generate", 
    should_continue, 
    {END: END, "reflect": "reflect"}
)

builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generate")
builder.add_edge("reflect", "research_critique")
builder.add_edge("research_critique", "generate")

graph = builder.compile(checkpointer=memory)

Creando la función generateEssay y la interfaz Gradio


In [ ]:
# app.py
import gradio as gr
from new_backend import graph # Importa el grafo de tu nuevo backend
import uuid

# --- Función que será llamada por Gradio para ejecutar el agente ---
def generate_essay(topic: str, max_revisions: int):
    # Ejecuta el grafo del agente para generar una redacción y transmite las salidas en tiempo real.
    thread_id = str(uuid.uuid4())
    thread_config = {"configurable": {"thread_id": thread_id}}

    initial_state = {
        "task": topic,
        "max_revisions": max_revisions,
        "revision_number": 0,
        "plan": "",
        "draft": "",
        "critique": "",
        "content": []
    }

    full_output = ""
    # Itera sobre el stream del grafo para obtener las salidas paso a paso
    for s in graph.stream(initial_state, thread_config):
        # La API de LangGraph devuelve un diccionario de diccionarios
        step_output = list(s.values())[0]

        # Formatea la salida para que sea más legible en la interfaz
        if "plan" in step_output:
            full_output += f"### �� Plan Generado:\n{step_output['plan']}\n\n"
        elif "content" in step_output:
            # Muestra el contenido de la investigación
            search_content = "\n".join(step_output['content'])
            full_output += f"### �� Contenido de Investigación:\n{search_content}\n\n"
        elif "draft" in step_output:
            full_output += f"### ✍️ Borrador Generado:\n{step_output['draft']}\n\n"
        elif "critique" in step_output:
            full_output += f"### �� Critica y Revisión:\n{step_output['critique']}\n\n"

        # Agrega una línea divisoria para separar los pasos
        full_output += "---\n" * 20 + "\n\n"
        
        yield full_output

Configurando la interfaz Gradio y ejecutando la aplicación


In [ ]:
# -- Creación de la Interfaz Gradio --
with gr.Blocks(theme=gr.themes.Default(spacing_size="sm", text_size="sm")) as demo:
    gr.Markdown("# �� Generador de Redacciones con Gemini y LangGraph")
    gr.Markdown(
        """
        Escribe el tema de tu redacción y el número de revisiones.
        El agente planificará, investigará, redactará y revisará el texto.
        """
    )
    with gr.Row():
        essay_topic = gr.Textbox(label="Tema de la Redacción", placeholder="Ej: La importancia de la inteligencia artificial en la educación")
        max_revisions_slider = gr.Slider(minimum=0, maximum=3, step=1, value=1, label="Número Máximo de Revisiones")
        generate_button = gr.Button("Generar Redacción", variant="primary")
    output_textbox = gr.Textbox(label="Proceso y Redacción Final", lines=20, max_lines=40)

    # Asocia el botón a la función Python
    generate_button.click(
        fn=generate_essay,
        inputs=[essay_topic, max_revisions_slider],
        outputs=output_textbox
    )

# Lanza la interfaz
if __name__ == "__main__":
    demo.launch(share=False)

Ejecutando la aplicación y generando redacciones


In [ ]:
pip install gradio
python app.py